In [6]:
# Join ml_enrollment_feature_table with dim_school to use SCHOOL_NAME instead of SCHOOL_KEY

from pyspark.sql import functions as F

# Directly query Lakehouse tables and join on SCHOOL_KEY
df = spark.sql("""
    SELECT
        s.SCHOOL_NAME,
        f.*
    FROM MDMF_COPY_RN.dbo.ml_enrollment_feature_table f
    LEFT JOIN MDMF_COPY_RN.dbo.dim_school s
        ON f.SCHOOL_KEY = s.SCHOOL_KEY
""")

# Remove SCHOOL_KEY now that we have SCHOOL_NAME
if "SCHOOL_KEY" in df.columns:
    df = df.drop("SCHOOL_KEY")
    df = df.drop("GRADE_NUMERIC")

display(df)

StatementMeta(, 7795a3f8-70e5-44b5-b988-ee619294d8b4, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 055ff04f-eed6-416f-bfd0-ea36f0db3cf1)

In [9]:
print(df.columns)

StatementMeta(, 7795a3f8-70e5-44b5-b988-ee619294d8b4, 11, Finished, Available, Finished, False)

['SCHOOL_NAME', 'GRADE', 'SCHOOL_YEAR', 'ENROLLMENT', 'SAME_GRADE_LAST_YEAR', 'SAME_GRADE_2YR_AGO', 'FEEDER_GRADE_LAST_YEAR', 'SCHOOL_TOTAL_ENROLLMENT']


In [14]:
# GBT-based forecasting of enrollment count per school + grade + year
# Target: enrollment count per school per grade per year
# Grain: SCHOOL_NAME, GRADE, SCHOOL_YEAR
# Horizon: forecast next 1 school year beyond the latest historical SCHOOL_YEAR

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# =====================================================================
# 0. CONFIGURATION
# =====================================================================

# Label and key columns based on your schema
LABEL_COL = "ENROLLMENT"           # numeric enrollment count per school+grade+year
TIME_COL = "SCHOOL_YEAR"          # time dimension (year)
SCHOOL_COL = "SCHOOL_NAME"        # school identifier
GRADE_COL = "GRADE"               # grade identifier

# Validate that the expected columns exist
required_cols = [LABEL_COL, TIME_COL, SCHOOL_COL, GRADE_COL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in df: {missing}.")

# Cast SCHOOL_YEAR to integer if it is not already
model_df = df.withColumn(TIME_COL, F.col(TIME_COL).cast(IntegerType()))

# Drop rows with nulls in key fields or label
model_df = model_df.dropna(subset=[LABEL_COL, TIME_COL, SCHOOL_COL, GRADE_COL])

# Fill remaining nulls in numeric feature columns with 0 to avoid NaN in feature vector
numeric_fill_cols = [
    "SAME_GRADE_LAST_YEAR",
    "SAME_GRADE_2YR_AGO",
    "FEEDER_GRADE_LAST_YEAR",
    "SCHOOL_TOTAL_ENROLLMENT",
]
model_df = model_df.fillna(0.0, subset=[c for c in numeric_fill_cols if c in model_df.columns])

# =====================================================================
# 1. TRAIN / VALIDATION SPLIT BY TIME
# =====================================================================

# Determine min/max year and define horizon
year_stats = model_df.select(
    F.min(TIME_COL).alias("min_year"),
    F.max(TIME_COL).alias("max_year")
).collect()[0]
min_year = year_stats["min_year"]
max_year = year_stats["max_year"]

print(f"Available SCHOOL_YEAR range: {min_year}–{max_year}")

# Train on all years <= max_year - 1, validate on max_year, forecast max_year + 1
val_year = max_year
train_max_year = max_year - 1
forecast_year = max_year + 1

print(f"Training up to SCHOOL_YEAR <= {train_max_year}")
print(f"Validation year: {val_year}")
print(f"Forecasting year: {forecast_year}")

train_df = model_df.filter(F.col(TIME_COL) <= train_max_year)
val_df = model_df.filter(F.col(TIME_COL) == val_year)

print(f"Train rows: {train_df.count():,}")
print(f"Validation rows: {val_df.count():,}")

# =====================================================================
# 2. FEATURE SELECTION & PIPELINE DEFINITION
# =====================================================================

# Identify candidate numeric feature columns automatically:
# - numeric type
# - not the label
# - not the time column

numeric_cols = [
    f.name for f in train_df.schema.fields
    if str(f.dataType).startswith("IntegerType")
    or str(f.dataType).startswith("LongType")
    or str(f.dataType).startswith("DoubleType")
    or str(f.dataType).startswith("FloatType")
]

numeric_feature_cols = [
    c for c in numeric_cols
    if c not in {LABEL_COL, TIME_COL}
]

# Categorical features to index
categorical_cols = [SCHOOL_COL, GRADE_COL]

if not numeric_feature_cols:
    raise ValueError(
        "No numeric feature columns were detected besides the label/time. "
        "Please add or identify numeric features to train the model."
    )

print("Numeric feature columns:", numeric_feature_cols)
print("Categorical columns:", categorical_cols)

# StringIndexers for categorical variables
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]

# Assemble all numeric features + indexed categoricals into a single feature vector
assembler_inputs = numeric_feature_cols + [f"{c}_idx" for c in categorical_cols]

feature_assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

# GBT Regressor configuration
# maxBins increased to handle high-cardinality SCHOOL_NAME (~1000+ distinct values)
gbt = GBTRegressor(
    labelCol=LABEL_COL,
    featuresCol="features",
    maxDepth=5,
    maxIter=100,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42,
    maxBins=2048
)

# Full pipeline
stages = []
stages.extend(indexers)
stages.append(feature_assembler)
stages.append(gbt)

pipeline = Pipeline(stages=stages)

# =====================================================================
# 3. MODEL TRAINING
# =====================================================================

print("\nFitting GBT model...")
model = pipeline.fit(train_df)
print("Model training complete.")

# =====================================================================
# 4. EVALUATION ON VALIDATION YEAR
# =====================================================================

val_pred = model.transform(val_df)

# Evaluate using RMSE, MAE, and R2
evaluators = {
    "rmse": RegressionEvaluator(labelCol=LABEL_COL, predictionCol="prediction", metricName="rmse"),
    "mae":  RegressionEvaluator(labelCol=LABEL_COL, predictionCol="prediction", metricName="mae"),
    "r2":   RegressionEvaluator(labelCol=LABEL_COL, predictionCol="prediction", metricName="r2"),
}

metrics = {name: ev.evaluate(val_pred) for name, ev in evaluators.items()}

print("\nValidation metrics (year = %s):" % val_year)
for k, v in metrics.items():
    print(f"  {k.upper()}: {v:.4f}")

print("\nSample validation predictions (actual vs predicted):")
val_sample = (
    val_pred
    .select(SCHOOL_COL, GRADE_COL, TIME_COL, LABEL_COL, "prediction")
    .orderBy(SCHOOL_COL, GRADE_COL, TIME_COL)
    .limit(50)
)
display(val_sample)

# =====================================================================
# 5. FORECAST NEXT 1 SCHOOL YEAR
# =====================================================================

# Strategy: use the latest available year per (school, grade) as the base
# and advance SCHOOL_YEAR by +1 to create future rows. All other features
# are taken as-is from the latest year. This assumes stability of features
# or that they encode appropriate lagged information.

w_latest = (
    Window.partitionBy(SCHOOL_COL, GRADE_COL)
          .orderBy(F.col(TIME_COL).desc())
)

latest_per_group = (
    model_df
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Create future DataFrame by copying latest rows and bumping the year
future_df = latest_per_group.withColumn(TIME_COL, F.lit(forecast_year))

print("\nGenerating forecast for SCHOOL_YEAR =", forecast_year)

future_pred = model.transform(future_df)

future_forecast = (
    future_pred
    .select(SCHOOL_COL, GRADE_COL, TIME_COL, "prediction")
    .orderBy(SCHOOL_COL, GRADE_COL)
)

print("Future enrollment forecasts per school + grade for year", forecast_year)
display(future_forecast)

# The `future_forecast` DataFrame is the final forecast output.


StatementMeta(, 7795a3f8-70e5-44b5-b988-ee619294d8b4, 16, Finished, Available, Finished, False)

Available SCHOOL_YEAR range: 2005–2026
Training up to SCHOOL_YEAR <= 2025
Validation year: 2026
Forecasting year: 2027
Train rows: 160,986
Validation rows: 9,690
Numeric feature columns: ['SAME_GRADE_LAST_YEAR', 'SAME_GRADE_2YR_AGO', 'FEEDER_GRADE_LAST_YEAR', 'SCHOOL_TOTAL_ENROLLMENT']
Categorical columns: ['SCHOOL_NAME', 'GRADE']

Fitting GBT model...


Model training complete.

Validation metrics (year = 2026):
  RMSE: 77.7569
  MAE: 13.1795
  R2: 0.4826

Sample validation predictions (actual vs predicted):


SynapseWidget(Synapse.DataFrame, 545617fb-1811-4ee6-95b9-1e3906fc5452)


Generating forecast for SCHOOL_YEAR = 2027
Future enrollment forecasts per school + grade for year 2027


SynapseWidget(Synapse.DataFrame, 5cf9e283-5082-4f84-820c-e0168982094d)